# UdaciHeadline: LLM Inference Optimization Project

## Project Introduction
Large Language Models (LLMs) are transforming content creation, but deploying them efficiently remains a major hurdle. Imagine you're an ML Engineer at a bustling online news portal. Your key task? Automatically generating catchy headlines from article summaries using an LLM. The problem? The current inference process is sluggish, causing publication delays and driving up operational costs. In this project, UdaciHeadline, you'll step into this role and tackle this critical challenge head-on. Your mission is to accelerate the headline generation pipeline significantly by applying state-of-the-art LLM inference optimization techniques. Get ready to dive deep into practical optimization and deployment!

## Project Summary
This project provides hands-on experience in optimizing the inference performance of a pre-trained Large Language Model (like Llama-3.2-1B) for news headline generation. You will bring together concepts of LLM architecture, optimization techniques, and deployment frameworks. Specifically, you will:

1.  **Establish a baseline** inference pipeline and profile its performance.
2.  Implement and evaluate architectural optimizations like **KV-caching**.
3.  Apply model compression techniques like **quantization** and **pruning**.
4.  Configure and benchmark **distributed inference** using Tensor and Pipeline Parallelism.
5.  Apply advanced decoding mechanisms like **speculative decoding**.
6.  Perform comprehensive **benchmarking and analysis** across all stages.
7.  Produce a **final report** summarizing findings and trade-offs.

## Imports and Global Configuration

Let's import the libraries we'll use throughout the project and define some constants like the model name and the prompt template.

In [1]:
import os
import sys
import gc
import json
import math
import time
import platform
import logging
import warnings
import subprocess
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import psutil
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load as load_metric
from pprint import pprint
import torch.nn.utils.prune as prune

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# ---- Constants ----
# Model resolution: the Udacity workspace ships Llama-3.2-1B locally; elsewhere use the ungated Hub mirror
# of the same weights (override with UDACI_MODEL / UDACI_TARGET_MODEL).
_LOCAL_1B = "/voc/shared/models/llama/Llama-3.2-1B"
_LOCAL_3B = "/voc/shared/models/llama/Llama-3.2-3B"
MODEL_NAME = os.environ.get("UDACI_MODEL", _LOCAL_1B if os.path.isdir(_LOCAL_1B) else "unsloth/Llama-3.2-1B")
TARGET_MODEL_NAME = os.environ.get("UDACI_TARGET_MODEL", _LOCAL_3B if os.path.isdir(_LOCAL_3B) else "unsloth/Llama-3.2-3B")
if os.path.isdir(MODEL_NAME):
    os.environ["HF_HUB_OFFLINE"] = "1"      # only go offline when everything is available locally

DATASET_PATH = os.environ.get("UDACI_DATASET", "../dataset/News_Category_Dataset.json")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 on GPU (Llama checkpoints are stored in bf16); fp32 on CPU (fastest CPU kernels, no bf16 emulation)
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

MAX_NEW_TOKENS = 24            # Max length for the generated headline (HuffPost headlines are ~10-15 tokens)
N_EVAL = int(os.environ.get("UDACI_N_EVAL", 20))   # evaluation samples per configuration
N_SHOT = 2                     # few-shot examples in the prompt (base model, not instruction-tuned)
SEED = 42
FORCE_RERUN = os.environ.get("UDACI_FORCE_RERUN", "0") == "1"   # ignore cached results/*.json

PROMPT = \
"""You are a news editor. Write a short, catchy headline for each article summary.

{examples}Summary: {summary}
Headline:"""

torch.manual_seed(SEED)
print(f"Device: {DEVICE} | dtype: {DTYPE} | model: {MODEL_NAME} | target (spec. decoding): {TARGET_MODEL_NAME}")
print(f"N_EVAL={N_EVAL}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}, N_SHOT={N_SHOT}, results -> {RESULTS_DIR.resolve()}")


Device: cpu | dtype: torch.float32 | model: unsloth/Llama-3.2-1B | target (spec. decoding): unsloth/Llama-3.2-3B
N_EVAL=20, MAX_NEW_TOKENS=24, N_SHOT=2, results -> /home/architect/UdaciHeadline/project/results


In [2]:
def environment_info():
    """Hardware/software details recorded for reproducibility."""
    import transformers, datasets, evaluate, accelerate
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "evaluate": evaluate.__version__,
        "accelerate": accelerate.__version__,
        "cpu": platform.processor() or "unknown",
        "cpu_count": os.cpu_count(),
        "torch_threads": torch.get_num_threads(),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "cuda_available": torch.cuda.is_available(),
        "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "device": DEVICE, "dtype": str(DTYPE), "model": MODEL_NAME, "target_model": TARGET_MODEL_NAME,
        "n_eval": N_EVAL, "max_new_tokens": MAX_NEW_TOKENS,
    }
    try:
        info["bitsandbytes"] = __import__("bitsandbytes").__version__
    except Exception:
        info["bitsandbytes"] = None
    try:  # nicer CPU name on Linux
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu"] = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    return info

ENV = environment_info()
json.dump(ENV, open(RESULTS_DIR / "environment.json", "w"), indent=2)
pprint(ENV)


{'accelerate': '1.14.0',
 'bitsandbytes': '0.50.1',
 'cpu': 'Intel(R) Core(TM) i7-10610U CPU @ 1.80GHz',
 'cpu_count': 8,
 'cuda_available': False,
 'datasets': '5.0.1',
 'device': 'cpu',
 'dtype': 'torch.float32',
 'evaluate': '0.4.6',
 'gpus': [],
 'max_new_tokens': 24,
 'model': 'unsloth/Llama-3.2-1B',
 'n_eval': 20,
 'platform': 'Linux-7.0.0-28-generic-x86_64-with-glibc2.39',
 'python': '3.12.3',
 'ram_total_gb': 33.3,
 'target_model': 'unsloth/Llama-3.2-3B',
 'torch': '2.13.0+cpu',
 'torch_threads': 4,
 'transformers': '5.15.0'}


## Data Loading

We will use the "News Category Dataset" from Kaggle. The `kagglehub` library makes it easy to download and access. Your task is to implement the function to load and preprocess the data according to the docstring.

In [3]:
def load_news_dataset(path, n_eval=N_EVAL, n_shot=N_SHOT, seed=SEED):
    """Load the News Category Dataset (JSON lines) with the HF `datasets` library and prepare it for
    headline generation.

    Steps
    1. `load_dataset("json")` -> ~210k HuffPost articles.
    2. Keep only articles that have both a headline and a reasonably informative short_description
       (>= 15 words) so the model has something to summarise, and drop very long ones (> 80 words) to keep
       prompts short.
    3. Shuffle with a fixed seed and split off `n_shot` few-shot examples (used inside the prompt) and
       `n_eval` evaluation samples.  The same samples are used for every configuration.
    Returns (eval_dataset, fewshot_examples) where the dataset has columns summary / headline / category.
    """
    raw = load_dataset("json", data_files=path, split="train")

    def _ok(ex):
        s, h = ex.get("short_description") or "", ex.get("headline") or ""
        n = len(s.split())
        return 15 <= n <= 80 and len(h.split()) >= 4

    ds = raw.filter(_ok)
    ds = ds.rename_column("short_description", "summary").select_columns(["summary", "headline", "category"])
    ds = ds.shuffle(seed=seed)
    fewshot = [ds[i] for i in range(n_shot)]
    eval_ds = ds.select(range(n_shot, n_shot + n_eval))
    print(f"Loaded {len(raw):,} articles, {len(ds):,} after filtering; using {n_shot} few-shot + {len(eval_ds)} eval samples.")
    return eval_ds, fewshot


eval_dataset, FEWSHOT = load_news_dataset(DATASET_PATH)
FEWSHOT_BLOCK = "".join(f"Summary: {ex['summary']}\nHeadline: {ex['headline']}\n\n" for ex in FEWSHOT)

def build_prompt(summary):
    return PROMPT.format(examples=FEWSHOT_BLOCK, summary=summary.strip())

print(build_prompt(eval_dataset[0]["summary"]))
print("\nReference headline:", eval_dataset[0]["headline"])


Loaded 209,527 articles, 127,270 after filtering; using 2 few-shot + 20 eval samples.
You are a news editor. Write a short, catchy headline for each article summary.

Summary: An end-of-life discussion is not a conversation likely to arise spontaneously on its own. Whether you are an aging parent or a concerned adult child, you must make the first move. Seize any opportunity to begin the conversation.
Headline: Hospice: Having an End-of-Life Conversation in the Midst of Life, Part 1

Summary: Some online deals are too easy to find and too hard to pass up. My new fondness for coupons started when I was visiting my folks in Florida. Before entering the Gap Outlet somewhere along the Gulf coast, I found an Internet coupon on the sidewalk: 50 percent off any purchase!
Headline: Learning From My Elders: How to Use Online Coupons

Summary: FYI, Americans: The BAFTAs are tomorrow, the same night as the Grammys. Watson, of course, is the face of Lancome; her most
Headline:

Reference headline:

# 2. Baseline Performance

Before we can optimize, we need a starting point. Here, you'll establish the baseline performance of the `Llama-3.2-1B` model without any specific optimizations. We will measure latency, throughput, and the quality of the generated headlines using the ROUGE score.

### Your Task: Implement the Evaluation Pipeline
You need to implement the core functions for loading a model, generating a headline, and evaluating performance. These functions will be reused for every optimization technique.

In [4]:
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def _peak_memory_gb():
    """Peak memory during the last measurement window: CUDA allocator peak on GPU, process RSS on CPU."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1e9
    return psutil.Process().memory_info().rss / 1e9

def model_footprint_gb(model):
    try:
        return model.get_memory_footprint() / 1e9
    except Exception:
        return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9


def load_model(model_name, quantization_config=None, device_map=None, dtype=None):
    """Load a tokenizer + causal LM.

    * `dtype` defaults to DTYPE (bf16 on GPU, fp32 on CPU).
    * `quantization_config` (BitsAndBytesConfig) enables 8/4-bit loading; bitsandbytes needs a device_map,
      so one is supplied automatically ("auto" on GPU, everything on CPU otherwise).
    * `device_map` ("auto", "balanced", or an explicit dict) hands placement to `accelerate` -- this is how
      tensor / pipeline parallel sharding across several GPUs is requested. Without it the model is moved to
      DEVICE as a whole.
    Returns (model, tokenizer). Padding side is set to left so batched generation works.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    kwargs = {"dtype": dtype or DTYPE, "low_cpu_mem_usage": True}
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config
        if device_map is None:
            device_map = "auto" if DEVICE == "cuda" else {"": "cpu"}
    if device_map is not None:
        kwargs["device_map"] = device_map

    t0 = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    if device_map is None:
        model.to(DEVICE)
    model.eval()
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print(f"Loaded {model_name} in {time.perf_counter()-t0:.1f}s | footprint {model_footprint_gb(model):.2f} GB"
          f" | dtype {next(model.parameters()).dtype} | device_map: {getattr(model, 'hf_device_map', None) or DEVICE}")
    return model, tokenizer


def _clean_headline(text):
    """First non-empty line of the continuation, without quotes/trailing junk."""
    for line in text.split("\n"):
        line = line.strip().strip('"').strip()
        if line:
            return line
    return text.strip()


def generate_headline(model, tokenizer, summary, generation_args):
    """Generate one headline and measure its latency.

    Returns (headline:str, latency_s:float, new_tokens:int). Timing brackets only `generate()` (tokenisation
    excluded) and is synchronised on CUDA. Generation stops at MAX_NEW_TOKENS, EOS or the first newline.
    """
    prompt = build_prompt(summary)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    args = dict(generation_args)
    args.setdefault("max_new_tokens", MAX_NEW_TOKENS)
    args.setdefault("do_sample", False)
    args.setdefault("pad_token_id", tokenizer.pad_token_id)
    # stop at the end of the headline line
    args.setdefault("stop_strings", ["\n"])
    args.setdefault("tokenizer", tokenizer)

    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, **args)
    _sync()
    latency = time.perf_counter() - t0

    new_ids = out[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    n_new = int((new_ids != tokenizer.pad_token_id).sum().item()) or new_ids.numel()
    return _clean_headline(text), latency, n_new


ROUGE = load_metric("rouge")

def report_metrics(results, latencies, max_new_tokens, label="", extra=None, verbose=True):
    """Compute and print latency / throughput / memory / ROUGE metrics.

    results   : list of dicts with keys headline, reference, new_tokens
    latencies : per-sample generate() latencies in seconds
    """
    lat = np.asarray(latencies, dtype=float)
    toks = np.asarray([r["new_tokens"] for r in results], dtype=float)
    # use_aggregator=False -> per-sample scores (the default bootstrap aggregation is randomised); we average them
    rouge = ROUGE.compute(predictions=[r["headline"] for r in results],
                          references=[r["reference"] for r in results], use_stemmer=True, use_aggregator=False)
    rouge = {k: float(np.mean(v)) for k, v in rouge.items()}
    m = dict(extra or {})          # extra info first, so freshly computed metrics below always win
    m.update({
        "label": label,
        "n_samples": len(results),
        "max_new_tokens": max_new_tokens,
        "latency_mean_s": float(lat.mean()),
        "latency_std_s": float(lat.std()),
        "latency_p50_s": float(np.percentile(lat, 50)),
        "latency_p99_s": float(np.percentile(lat, 99)),
        "latency_min_s": float(lat.min()),
        "latency_max_s": float(lat.max()),
        "total_time_s": float(lat.sum()),
        "avg_new_tokens": float(toks.mean()),
        "throughput_tok_s": float(toks.sum() / lat.sum()),          # generated tokens per second
        "throughput_samples_s": float(len(results) / lat.sum()),    # headlines per second
        "rouge1": float(rouge["rouge1"]), "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]), "rougeLsum": float(rouge["rougeLsum"]),
    })
    if verbose:
        print(f"\n=== {label or 'metrics'} ===")
        print(f"  samples            : {m['n_samples']}  (avg {m['avg_new_tokens']:.1f} new tokens, cap {max_new_tokens})")
        print(f"  latency mean / p50 / p99 : {m['latency_mean_s']:.3f} / {m['latency_p50_s']:.3f} / {m['latency_p99_s']:.3f} s")
        print(f"  throughput         : {m['throughput_tok_s']:.2f} tokens/s  ({m['throughput_samples_s']*60:.1f} headlines/min)")
        if "peak_memory_gb" in m:
            print(f"  memory             : model {m.get('model_footprint_gb', float('nan')):.2f} GB, peak {m['peak_memory_gb']:.2f} GB ({m.get('memory_kind','')})")
        print(f"  ROUGE-1 / -2 / -L  : {m['rouge1']:.4f} / {m['rouge2']:.4f} / {m['rougeL']:.4f}")
    return m


def evaluate_model(dataset, model, tokenizer, generation_args, n=N_EVAL, label="run", extra=None,
                   warmup=True, save=True, verbose=True, cache=True):
    """Run headline generation over the first `n` samples and report metrics.

    * one warm-up generation (kernel/JIT/page-in) that is not timed
    * peak-memory stats reset before the loop and read after it
    * results are stored under results/<label>.json (and re-loaded on later runs unless FORCE_RERUN=1)
    """
    path = RESULTS_DIR / f"{label}.json"
    if cache and not FORCE_RERUN and path.exists():
        m = json.load(open(path))
        print(f"[cache] loaded {path}  (set UDACI_FORCE_RERUN=1 to recompute)")
        # recompute the derived metrics from the stored samples (keeps ROUGE deterministic across runs)
        m.update(report_metrics(m["samples"], [s["latency_s"] for s in m["samples"]], m["max_new_tokens"], label=label,
                                extra={k: v for k, v in m.items() if k not in ("samples",)}, verbose=verbose))
        json.dump(m, open(path, "w"), indent=2)
        return m

    n = min(n, len(dataset))
    if warmup:
        generate_headline(model, tokenizer, dataset[0]["summary"], generation_args)
    gc.collect(); _reset_peak_memory()
    rss_before = psutil.Process().memory_info().rss / 1e9

    results, latencies = [], []
    t_start = time.perf_counter()
    for i in range(n):
        ex = dataset[i]
        headline, lat, n_new = generate_headline(model, tokenizer, ex["summary"], generation_args)
        results.append({"headline": headline, "reference": ex["headline"], "new_tokens": n_new, "latency_s": lat})
        latencies.append(lat)
        if verbose and (i < 3 or i == n - 1):
            print(f"  [{i+1:>2}/{n}] {lat:6.2f}s {n_new:>2} tok | gen: {headline[:70]!r}\n{'':14}| ref: {ex['headline'][:70]!r}")
    wall = time.perf_counter() - t_start

    mem_extra = {
        "model_footprint_gb": model_footprint_gb(model),
        "peak_memory_gb": _peak_memory_gb(),
        "memory_kind": "cuda max_memory_allocated" if DEVICE == "cuda" else "process RSS",
        "rss_delta_gb": psutil.Process().memory_info().rss / 1e9 - rss_before,
        "wall_time_s": wall,
        "generation_args": {k: (str(v) if not isinstance(v, (int, float, bool, str, type(None))) else v)
                            for k, v in generation_args.items() if k != "tokenizer"},
        "device": DEVICE, "dtype": str(next(model.parameters()).dtype),
    }
    if extra:
        mem_extra.update(extra)
    m = report_metrics(results, latencies, generation_args.get("max_new_tokens", MAX_NEW_TOKENS), label=label,
                       extra=mem_extra, verbose=verbose)
    m["samples"] = results
    if save:
        json.dump(m, open(path, "w"), indent=2)
        print(f"  saved -> {path}")
    return m


def show_headlines(metrics, k=5):
    """Pretty-print k generated vs reference headlines."""
    rows = [{"generated": s["headline"], "reference": s["reference"], "tokens": s["new_tokens"],
             "latency_s": round(s["latency_s"], 2)} for s in metrics["samples"][:k]]
    with pd.option_context("display.max_colwidth", 90, "display.width", 200):
        display(pd.DataFrame(rows))


In [5]:
# ---- Baseline: plain autoregressive decoding WITHOUT the KV cache ----
# Every decoding step re-computes attention over the whole prefix, i.e. O(n^2) work per headline.
model, tokenizer = load_model(MODEL_NAME)

baseline_args = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "use_cache": False}
baseline_metrics = evaluate_model(eval_dataset, model, tokenizer, baseline_args, n=N_EVAL, label="baseline_no_cache")
show_headlines(baseline_metrics)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded unsloth/Llama-3.2-1B in 1.3s | footprint 4.94 GB | dtype torch.float32 | device_map: cpu
[cache] loaded results/baseline_no_cache.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== baseline_no_cache ===
  samples            : 20  (avg 11.9 new tokens, cap 24)
  latency mean / p50 / p99 : 58.388 / 54.896 / 109.534 s
  throughput         : 0.20 tokens/s  (1.0 headlines/min)
  memory             : model 4.94 GB, peak 6.20 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1545 / 0.0281 / 0.1445


,generated,reference,tokens,latency_s
0,The BAFTAs: A Night of Glamour and Glamourous Stars,Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS),16,81.15
1,The End of Life: A Personal Journey,The Conversation Nobody Wants To Have -- But Should,9,45.84
2,The Inking of the Inkblot,"Republicans Sprinting Toward Tax Cuts, Deficits Be Damned",9,39.47
3,Heart Disease: A Reminder to Be Wary of Symptoms,A New Dimension to Christmas Leads to Year-Round Devotion,12,55.37
4,Antibiotic Use in Hospitals: A Call for Appropriate Use,"1 In 25 Patients Experience Infection Related To Hospital Stay, Report Shows",13,54.42


In [6]:
# ---- Optional deeper look: PyTorch profiler on one baseline generation (top operators) ----
import torch.profiler
_ex = eval_dataset[0]["summary"]
_acts = [torch.profiler.ProfilerActivity.CPU] + ([torch.profiler.ProfilerActivity.CUDA] if DEVICE == "cuda" else [])
with torch.profiler.profile(activities=_acts, profile_memory=True) as prof:
    with torch.profiler.record_function("baseline_generate"):
        generate_headline(model, tokenizer, _ex, baseline_args)
sort_key = "self_cuda_time_total" if DEVICE == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=8))
prof.export_chrome_trace(str(RESULTS_DIR / "baseline_trace.json"))


USDT:2026-08-16 17:18:07 668145:668145 SyncActivityProfilerHandler.cpp:52] profiler_start


USDT:2026-08-16 17:19:02 668145:668145 SyncActivityProfilerHandler.cpp:59] profiler_stop


-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                             aten::mm        93.27%       51.457s        93.27%       51.459s      28.462ms       4.74 GB       4.74 GB          1808  
                                    baseline_generate         2.01%        1.110s       100.00%       55.171s       55.171s      16.63 MB     -14.15 GB             1  
    aten::_scaled_dot_product_flash_attention_for_cpu         1.96%        1.079s         1.98%        1.091s       4.261ms     427.58 MB     -69.12 MB         

# 3. Architectural Optimization: KV Caching

**Your Task:** One of the most effective ways to speed up token generation is using a Key-Value (KV) cache. This avoids re-computing attention scores for tokens that are already part of the sequence. Enable the `use_cache` flag in the generation arguments and re-run the evaluation. Observe the impact on latency and throughput.

In [7]:
# ---- KV caching: keys/values of previous tokens are cached, each step only computes attention for the new token ----
kv_args = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "use_cache": True}
kv_metrics = evaluate_model(eval_dataset, model, tokenizer, kv_args, n=N_EVAL, label="kv_cache")

def compare(a, b, name_a="baseline", name_b="optimized"):
    rows = []
    for key, nice in [("latency_mean_s", "Mean latency (s)"), ("latency_p99_s", "P99 latency (s)"),
                      ("throughput_tok_s", "Throughput (tok/s)"), ("peak_memory_gb", "Peak memory (GB)"),
                      ("model_footprint_gb", "Model footprint (GB)"),
                      ("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L")]:
        va, vb = a.get(key, float("nan")), b.get(key, float("nan"))
        rows.append({"metric": nice, name_a: round(va, 4), name_b: round(vb, 4),
                     "change": f"{(vb/va - 1)*100:+.1f}%" if va else ""})
    display(pd.DataFrame(rows).set_index("metric"))

compare(baseline_metrics, kv_metrics, "baseline (no cache)", "KV cache")
same = sum(a["headline"] == b["headline"] for a, b in zip(baseline_metrics["samples"], kv_metrics["samples"]))
print(f"Identical headlines with/without cache: {same}/{len(kv_metrics['samples'])} "
      "(the cache changes arithmetic order slightly, so tiny differences are possible)")


[cache] loaded results/kv_cache.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== kv_cache ===
  samples            : 20  (avg 11.9 new tokens, cap 24)
  latency mean / p50 / p99 : 6.504 / 6.488 / 9.064 s
  throughput         : 1.83 tokens/s  (9.2 headlines/min)
  memory             : model 4.94 GB, peak 6.26 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1545 / 0.0281 / 0.1445


,baseline (no cache),KV cache,change
metric,,,
Mean latency (s),58.3877,6.5043,-88.9%
P99 latency (s),109.5342,9.0636,-91.7%
Throughput (tok/s),0.2038,1.8296,+797.7%
Peak memory (GB),6.2003,6.2640,+1.0%
Model footprint (GB),4.9433,4.9433,+0.0%
ROUGE-1,0.1545,0.1545,+0.0%
ROUGE-2,0.0281,0.0281,+0.0%
ROUGE-L,0.1445,0.1445,+0.0%


Identical headlines with/without cache: 20/20 (the cache changes arithmetic order slightly, so tiny differences are possible)


# 4. Model Compression: Pruning

**Your Task:** Pruning removes redundant model weights, which can reduce model size and potentially speed up inference. Here, you will implement unstructured, magnitude-based pruning by creating a function that applies it to the model's linear layers and then evaluating the result.

In [8]:
def prune_model_weights(model, amount=0.3, layer_types=(torch.nn.Linear,), skip=("lm_head",), verbose=True):
    """Apply L1-unstructured (magnitude) pruning to every Linear layer of the model, in place.

    For each layer the `amount` fraction of weights with the smallest |w| is zeroed. Pruning is made
    permanent right away with `prune.remove` (which folds weight_orig * mask back into .weight and drops the
    forward hook) so that only ONE extra weight-sized mask exists at any time -- important on memory-bound
    machines. Layers whose name contains one of `skip` are left intact (the LM head is tied to the embeddings
    in Llama-3.2, and pruning it would also prune the input embeddings).

    Returns a dict with the number of pruned layers, global sparsity and any per-layer errors encountered.
    """
    pruned, errors, zeros, total = [], [], 0, 0
    t0 = time.perf_counter()
    for name, module in model.named_modules():
        if not isinstance(module, layer_types) or any(s in name for s in skip):
            continue
        try:
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")                        # make permanent, free the mask
            w = module.weight.detach()
            zeros += int((w == 0).sum().item()); total += w.numel()
            pruned.append(name)
        except Exception as e:                                    # e.g. OOM on the mask, unsupported dtype
            errors.append({"layer": name, "error": f"{type(e).__name__}: {e}"})
            if verbose:
                print(f"  !! could not prune {name}: {type(e).__name__}: {e}")
    info = {"amount": amount, "pruned_layers": len(pruned), "errors": errors,
            "global_sparsity": zeros / total if total else 0.0, "pruning_time_s": time.perf_counter() - t0}
    if verbose:
        print(f"Pruned {len(pruned)} Linear layers to {amount:.0%} sparsity each in {info['pruning_time_s']:.1f}s "
              f"-> global sparsity over pruned weights {info['global_sparsity']:.2%}; {len(errors)} errors")
    return info


# Prune the model that is already loaded (KV cache stays enabled so this isolates the effect of pruning)
prune_info = prune_model_weights(model, amount=0.3)
pruned_metrics = evaluate_model(eval_dataset, model, tokenizer, kv_args, n=N_EVAL, label="pruned_30",
                                extra={"pruning": prune_info})
compare(kv_metrics, pruned_metrics, "KV cache (dense)", "KV cache + 30% pruning")
show_headlines(pruned_metrics)

# Sanity check on the tensors: the weights really are 30% zeros, still dense tensors of the same shape/dtype
_w = model.model.layers[0].mlp.gate_proj.weight
print(f"layer0.mlp.gate_proj: shape {tuple(_w.shape)}, dtype {_w.dtype}, sparsity {(_w == 0).float().mean():.3f}, "
      f"is_sparse={_w.is_sparse}")

# free the pruned model before loading the quantized ones
del model; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()


Pruned 112 Linear layers to 30% sparsity each in 49.0s -> global sparsity over pruned weights 30.00%; 0 errors
[cache] loaded results/pruned_30.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== pruned_30 ===
  samples            : 20  (avg 14.0 new tokens, cap 24)
  latency mean / p50 / p99 : 7.148 / 6.973 / 9.754 s
  throughput         : 1.96 tokens/s  (8.4 headlines/min)
  memory             : model 4.94 GB, peak 7.27 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1106 / 0.0154 / 0.1064


,KV cache (dense),KV cache + 30% pruning,change
metric,,,
Mean latency (s),6.5043,7.1484,+9.9%
P99 latency (s),9.0636,9.7536,+7.6%
Throughput (tok/s),1.8296,1.9585,+7.0%
Peak memory (GB),6.2640,7.2726,+16.1%
Model footprint (GB),4.9433,4.9433,+0.0%
ROUGE-1,0.1545,0.1106,-28.4%
ROUGE-2,0.0281,0.0154,-45.3%
ROUGE-L,0.1445,0.1064,-26.4%


,generated,reference,tokens,latency_s
0,Learning from My Elders: How to Use Online Coupons,Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS),12,6.14
1,Learning from My Elders: How to Use Online Coupons,The Conversation Nobody Wants To Have -- But Should,12,7.26
2,The 10th Amendment: The 10th Amendment,"Republicans Sprinting Toward Tax Cuts, Deficits Be Damned",12,6.20
3,The Heart of a Father: How to Use the Stories of My Elders to Help You,A New Dimension to Christmas Leads to Year-Round Devotion,19,9.01
4,The 2019 Call for Appropriate Use of Antibiotics in the Hospital,"1 In 25 Patients Experience Infection Related To Hospital Stay, Report Shows",16,7.39


layer0.mlp.gate_proj: shape (8192, 2048), dtype torch.float32, sparsity 0.300, is_sparse=False


# 5. Model Compression: Quantization

**Your Task:** Quantization reduces the precision of model weights (e.g., from 16-bit to 4-bit), significantly cutting down memory usage and often speeding up inference. You will define a 4-bit quantization configuration and use it to load and evaluate a new model.

In [9]:
def validate_quantized_model(model, tokenizer, name):
    """Basic functional checks for a quantized model: quantized modules exist, a forward pass gives finite
    logits, and greedy generation produces non-empty text."""
    q_layers = [n for n, m in model.named_modules() if type(m).__name__ in ("Linear8bitLt", "Linear4bit")]
    ex = eval_dataset[0]["summary"]
    with torch.no_grad():
        logits = model(**tokenizer(build_prompt(ex), return_tensors="pt").to(model.device)).logits
    finite = bool(torch.isfinite(logits.float()).all())
    headline, lat, n_new = generate_headline(model, tokenizer, ex, kv_args)
    ok = bool(q_layers) and finite and len(headline) > 0
    print(f"[{name}] quantized Linear layers: {len(q_layers)} | finite logits: {finite} | "
          f"sample headline ({n_new} tok, {lat:.2f}s): {headline!r} -> {'PASS' if ok else 'FAIL'}")
    return {"quantized_layers": len(q_layers), "finite_logits": finite, "sample_headline": headline, "pass": ok}


quant_configs = {
    # 8-bit LLM.int8(): int8 weights with fp16 outlier decomposition
    "quant_int8": BitsAndBytesConfig(load_in_8bit=True),
    # 4-bit NF4 with double quantization (QLoRA-style); compute in bf16 on GPU / fp32 on CPU
    "quant_nf4": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                    bnb_4bit_compute_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32),
}
quant_metrics = {}
for qname, qcfg in quant_configs.items():
    print(f"\n##### {qname} #####")
    try:
        qmodel, qtok = load_model(MODEL_NAME, quantization_config=qcfg)
        checks = validate_quantized_model(qmodel, qtok, qname)
        quant_metrics[qname] = evaluate_model(eval_dataset, qmodel, qtok, kv_args, n=N_EVAL, label=qname,
                                              extra={"validation": checks, "quantization": qcfg.to_dict()})
        show_headlines(quant_metrics[qname], k=3)
        del qmodel; gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    except Exception as e:
        # bitsandbytes needs CUDA (or a recent CPU backend); record the failure instead of crashing the notebook
        print(f"!! {qname} failed: {type(e).__name__}: {e}")
        quant_metrics[qname] = {"label": qname, "error": f"{type(e).__name__}: {e}"}
        json.dump(quant_metrics[qname], open(RESULTS_DIR / f"{qname}.json", "w"), indent=2)

if "quant_int8" in quant_metrics and "error" not in quant_metrics["quant_int8"]:
    compare(kv_metrics, quant_metrics["quant_int8"], "KV cache (fp32/bf16)", "8-bit")
if "quant_nf4" in quant_metrics and "error" not in quant_metrics["quant_nf4"]:
    compare(kv_metrics, quant_metrics["quant_nf4"], "KV cache (fp32/bf16)", "4-bit NF4")



##### quant_int8 #####


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded unsloth/Llama-3.2-1B in 5.9s | footprint 2.02 GB | dtype torch.float32 | device_map: cpu


[quant_int8] quantized Linear layers: 112 | finite logits: True | sample headline (16 tok, 22.48s): 'The BAFTAs: A Night of Glamour and Glamourous Stars' -> PASS
[cache] loaded results/quant_int8.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== quant_int8 ===
  samples            : 20  (avg 12.3 new tokens, cap 24)
  latency mean / p50 / p99 : 18.236 / 17.844 / 29.962 s
  throughput         : 0.67 tokens/s  (3.3 headlines/min)
  memory             : model 2.02 GB, peak 3.92 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1720 / 0.0340 / 0.1643


,generated,reference,tokens,latency_s
0,The BAFTAs: A Night of Glamour and Glamourous Stars,Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS),16,22.35
1,The End of Life: A Story of Loss and Rebirth,The Conversation Nobody Wants To Have -- But Should,13,19.66
2,The Inability to See the Forest for the Trees,"Republicans Sprinting Toward Tax Cuts, Deficits Be Damned",11,17.60


...

Loaded unsloth/Llama-3.2-1B in 13.5s | footprint 1.54 GB | dtype torch.float32 | device_map: cpu


[quant_nf4] quantized Linear layers: 112 | finite logits: True | sample headline (11 tok, 12.31s): 'The BAFTAs: The Face of Lancome' -> PASS
[cache] loaded results/quant_nf4.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== quant_nf4 ===
  samples            : 20  (avg 12.6 new tokens, cap 24)
  latency mean / p50 / p99 : 13.415 / 12.748 / 20.170 s
  throughput         : 0.94 tokens/s  (4.5 headlines/min)
  memory             : model 1.54 GB, peak 3.91 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1228 / 0.0231 / 0.1228


,generated,reference,tokens,latency_s
0,The BAFTAs: The Face of Lancome,Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS),11,12.11
1,The Death of My Father: A Personal Essay,The Conversation Nobody Wants To Have -- But Should,10,11.57
2,The Injustice of the Injustice: The Case of the Injustice of the Injustice,"Republicans Sprinting Toward Tax Cuts, Deficits Be Damned",19,17.72


,KV cache (fp32/bf16),8-bit,change
metric,,,
Mean latency (s),6.5043,18.2361,+180.4%
P99 latency (s),9.0636,29.9622,+230.6%
Throughput (tok/s),1.8296,0.6745,-63.1%
Peak memory (GB),6.2640,3.9209,-37.4%
Model footprint (GB),4.9433,2.0240,-59.1%
ROUGE-1,0.1545,0.1720,+11.3%
ROUGE-2,0.0281,0.0340,+20.9%
ROUGE-L,0.1445,0.1643,+13.7%


,KV cache (fp32/bf16),4-bit NF4,change
metric,,,
Mean latency (s),6.5043,13.4153,+106.3%
P99 latency (s),9.0636,20.1697,+122.5%
Throughput (tok/s),1.8296,0.9392,-48.7%
Peak memory (GB),6.2640,3.9128,-37.5%
Model footprint (GB),4.9433,1.5375,-68.9%
ROUGE-1,0.1545,0.1228,-20.5%
ROUGE-2,0.0281,0.0231,-17.8%
ROUGE-L,0.1445,0.1228,-15.0%


# 6. Distributed Inference (Multi-GPU)

**Your Task:** If you have multiple GPUs, you can split the model across them to reduce the memory burden on a single GPU and potentially improve latency. We will explore two common techniques: Tensor Parallelism and Pipeline Parallelism.

*Note: This section requires a multi-GPU environment.*

### Tensor Parallelism
Tensor parallelism splits individual model layers (the tensors) across multiple GPUs. Operations like matrix multiplications are executed in parallel on different GPUs, and the results are aggregated. This is highly effective for reducing the memory footprint of very large layers. The `accelerate` library can handle this automatically via `device_map="auto"`.

### Pipeline Parallelism
Pipeline parallelism assigns entire layers or blocks of layers to different GPUs, creating a sequence or "pipeline" that the data flows through. For example, layers 1-10 run on GPU 0, layers 11-20 run on GPU 1, and so on. This is useful for very deep models where even a single layer might be too large for one GPU after tensor parallelism.

In [10]:
from accelerate import infer_auto_device_map, init_empty_weights
from transformers import AutoConfig

N_GPUS = torch.cuda.device_count()
print(f"GPUs visible: {N_GPUS}")

def describe_device_map(dm):
    """Summarise a device_map as {device: [first..last layer]}"""
    by_dev = {}
    for k, v in dm.items():
        by_dev.setdefault(str(v), []).append(k)
    for dev, mods in by_dev.items():
        layers = sorted(int(m.split(".")[2]) for m in mods if m.startswith("model.layers.") and m.count(".") == 2)
        others = [m for m in mods if not m.startswith("model.layers.")]
        span = f"layers {layers[0]}-{layers[-1]}" if layers else ""
        print(f"  {dev:>6}: {len(mods):>3} modules  {span}  {others}")

# Results from a real multi-GPU run of project/distributed_benchmark.py (e.g. SageMaker) take precedence
_multi = RESULTS_DIR / "distributed_multigpu.json"
if _multi.exists():
    _mg = json.load(open(_multi))
    print(f"Found multi-GPU results from {_mg.get('environment', {}).get('gpus')}:")
    for k in ("tensor_parallel", "pipeline_parallel"):
        if k in _mg:
            print(f"  {k}: mean latency {_mg[k]['latency_mean_s']:.3f}s, {_mg[k]['throughput_tok_s']:.2f} tok/s, ROUGE-1 {_mg[k]['rouge1']:.4f}")
            json.dump(_mg[k], open(RESULTS_DIR / f"{k}.json", "w"), indent=2)

if N_GPUS >= 2:
    # ---- Tensor-parallel style sharding: accelerate spreads the weights over all GPUs ("auto") ----
    tp_model, tp_tok = load_model(MODEL_NAME, device_map="auto")
    describe_device_map(getattr(tp_model, "hf_device_map", {"": DEVICE}))
    tp_metrics = evaluate_model(eval_dataset, tp_model, tp_tok, kv_args, n=N_EVAL, label="tensor_parallel",
                                extra={"device_map": {k: str(v) for k, v in getattr(tp_model, "hf_device_map", {"": DEVICE}).items()}, "n_gpus": N_GPUS})
    del tp_model; gc.collect(); torch.cuda.empty_cache()
else:
    # ---- Simulation on a single device ----
    # We still go through accelerate's device_map machinery (this is exactly the code path used on
    # multi-GPU) but with only one device available everything lands on it, so the timing equals a
    # single-device run. The partition that WOULD be used on 2 GPUs is computed on the meta device below.
    print("\n< 2 GPUs: running the device_map='auto' code path on a single device (simulation).")
    cfg = AutoConfig.from_pretrained(MODEL_NAME)
    with init_empty_weights():
        _meta = AutoModelForCausalLM.from_config(cfg, dtype=torch.bfloat16)
    sim_map = infer_auto_device_map(_meta, max_memory={0: "1.6GiB", 1: "1.6GiB"}, no_split_module_classes=["LlamaDecoderLayer"])
    print("device_map='auto' would place the bf16 model on 2 x 1.6 GiB GPUs like this (tensor-parallel style split):")
    describe_device_map(sim_map)
    del _meta

    tp_model, tp_tok = load_model(MODEL_NAME, device_map="auto")
    tp_metrics = evaluate_model(eval_dataset, tp_model, tp_tok, kv_args, n=N_EVAL, label="tensor_parallel_sim",
                                extra={"simulated": True, "device_map": {k: str(v) for k, v in getattr(tp_model, "hf_device_map", {"": DEVICE}).items()},
                                       "would_be_device_map_2gpu": {k: str(v) for k, v in sim_map.items()}, "n_gpus": N_GPUS})
    del tp_model; gc.collect()


GPUs visible: 0

< 2 GPUs: running the device_map='auto' code path on a single device (simulation).


device_map='auto' would place the bf16 model on 2 x 1.6 GiB GPUs like this (tensor-parallel style split):
       0:   6 modules  layers 0-4  ['model.embed_tokens']
       1:  13 modules  layers 5-15  ['model.norm', 'model.rotary_emb']
    disk:   1 modules    ['lm_head']


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded unsloth/Llama-3.2-1B in 1.1s | footprint 4.94 GB | dtype torch.float32 | device_map: cpu


  [ 1/20]   7.24s 16 tok | gen: 'The BAFTAs: A Night of Glamour and Glamourous Stars'
              | ref: "Emma Watson's Sheer Pink Frock: Yay Or Nay? (PHOTOS)"


  [ 2/20]   6.81s  9 tok | gen: 'The End of Life: A Personal Journey'
              | ref: 'The Conversation Nobody Wants To Have -- But Should'


  [ 3/20]   5.33s  9 tok | gen: 'The Inking of the Inkblot'
              | ref: 'Republicans Sprinting Toward Tax Cuts, Deficits Be Damned'


  [20/20]   6.72s 12 tok | gen: "Indiana's Religious Freedom Law: A Threat to the Constitution"
              | ref: 'Indiana Takes on America: Discrimination Against Gays, Religious Freed'

=== tensor_parallel_sim ===
  samples            : 20  (avg 11.9 new tokens, cap 24)
  latency mean / p50 / p99 : 6.573 / 6.451 / 9.967 s
  throughput         : 1.81 tokens/s  (9.1 headlines/min)
  memory             : model 4.94 GB, peak 7.87 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1545 / 0.0281 / 0.1445
  saved -> results/tensor_parallel_sim.json


In [11]:
def pipeline_device_map(model_name, n_stages):
    """Explicit pipeline-parallel map: embeddings + first half of the decoder layers on device 0,
    second half + final norm + LM head on device 1 (generalises to n_stages)."""
    cfg = AutoConfig.from_pretrained(model_name)
    L = cfg.num_hidden_layers
    dm = {"model.embed_tokens": 0, "model.rotary_emb": 0}
    per = math.ceil(L / n_stages)
    for i in range(L):
        dm[f"model.layers.{i}"] = min(i // per, n_stages - 1)
    dm["model.norm"] = n_stages - 1
    dm["lm_head"] = n_stages - 1
    return dm

if N_GPUS >= 2:
    # ---- Pipeline parallelism: contiguous blocks of layers per GPU; activations hop from GPU to GPU ----
    pp_map = pipeline_device_map(MODEL_NAME, N_GPUS)
    describe_device_map(pp_map)
    pp_model, pp_tok = load_model(MODEL_NAME, device_map=pp_map)      # or device_map="balanced"
    pp_metrics = evaluate_model(eval_dataset, pp_model, pp_tok, kv_args, n=N_EVAL, label="pipeline_parallel",
                                extra={"device_map": {k: str(v) for k, v in pp_map.items()}, "n_gpus": N_GPUS})
    del pp_model; gc.collect(); torch.cuda.empty_cache()
    compare(tp_metrics, pp_metrics, "tensor parallel (auto)", "pipeline parallel (layer map)")
else:
    print("< 2 GPUs: showing the pipeline map for 2 stages and running it on the single available device (simulation).")
    pp_map = pipeline_device_map(MODEL_NAME, 2)
    describe_device_map(pp_map)
    # On one device every stage maps to the same device
    single = {k: (DEVICE if DEVICE == "cpu" else 0) for k in pp_map}
    pp_model, pp_tok = load_model(MODEL_NAME, device_map=single)
    pp_metrics = evaluate_model(eval_dataset, pp_model, pp_tok, kv_args, n=N_EVAL, label="pipeline_parallel_sim",
                                extra={"simulated": True, "would_be_device_map_2gpu": {k: str(v) for k, v in pp_map.items()}, "n_gpus": N_GPUS})
    del pp_model; gc.collect()
    compare(tp_metrics, pp_metrics, "tensor parallel (sim)", "pipeline parallel (sim)")


< 2 GPUs: showing the pipeline map for 2 stages and running it on the single available device (simulation).


       0:  10 modules  layers 0-7  ['model.embed_tokens', 'model.rotary_emb']
       1:  10 modules  layers 8-15  ['model.norm', 'lm_head']


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded unsloth/Llama-3.2-1B in 1.2s | footprint 4.94 GB | dtype torch.float32 | device_map: cpu
[cache] loaded results/pipeline_parallel_sim.json  (set UDACI_FORCE_RERUN=1 to recompute)

=== pipeline_parallel_sim ===
  samples            : 20  (avg 11.9 new tokens, cap 24)
  latency mean / p50 / p99 : 6.570 / 6.236 / 9.584 s
  throughput         : 1.81 tokens/s  (9.1 headlines/min)
  memory             : model 4.94 GB, peak 7.83 GB (process RSS)
  ROUGE-1 / -2 / -L  : 0.1545 / 0.0281 / 0.1445


,tensor parallel (sim),pipeline parallel (sim),change
metric,,,
Mean latency (s),6.5735,6.5703,-0.0%
P99 latency (s),9.9669,9.5837,-3.8%
Throughput (tok/s),1.8103,1.8112,+0.0%
Peak memory (GB),7.8729,7.8299,-0.5%
Model footprint (GB),4.9433,4.9433,+0.0%
ROUGE-1,0.1545,0.1545,+0.0%
ROUGE-2,0.0281,0.0281,+0.0%
ROUGE-L,0.1445,0.1445,+0.0%


# 7. Advanced Decoding: Speculative Decoding

**Your Task:** Speculative decoding uses a smaller, faster "draft" model to generate several candidate tokens. A larger, more accurate "target" model then verifies these tokens in a single forward pass. This can significantly speed up generation if the draft model is a good predictor. You will load a larger target model and a smaller draft model, benchmark the target model alone, and then benchmark it with assistance from the draft model.

In [12]:
# TODO: Implement and evaluate speculative decoding.

# 8. Final Report and Analysis

**Your Task:** Consolidate your findings into a summary report. 

1.  Fill in the Markdown table below with the **Latency**, **Throughput**, and **ROUGE scores** for each optimization technique you implemented.
2. Compile the final Project Report in PDF format:
    *   Document the entire process, detailing the methodology, techniques, and libraries used.
    *   Present the final benchmark results clearly.
    *   Provide a thorough analysis of the trade-offs between performance, resources, and quality for each optimization step.
    *   Conclude with recommendations for the most effective optimization strategy for this specific headline generation task, supported by your data.

Some example questions for discussing the trade-offs:
    *   Which method gave the best performance improvement?
    *   Did any methods significantly hurt the ROUGE score (quality)?
    *   Which optimization would you recommend for deployment in a production environment at the news portal, and why? Consider factors like cost, complexity, and performance.

## Performance Comparison

| Optimization Technique | Mean Latency (s) | Throughput (tokens/s) | ROUGE-1 Score |
|--------------------------|------------------|-----------------------|---------------|
| Baseline (No Cache)      | TODO             | TODO                  | TODO          |
| KV Caching               | TODO             | TODO                  | TODO          |
| Pruning (30%)            | TODO             | TODO                  | TODO          |
| Quantization (4-bit)     | TODO             | TODO                  | TODO          |
| Tensor Parallelism       | TODO             | TODO                  | TODO          |
| Pipeline Parallelism     | TODO             | TODO                  | TODO          |
| Speculative Decoding     | TODO             | TODO                  | TODO          |

---

